#**Домашнее задание №2**
##**Выполнила:** Шумилова Мария Константиновна 409920 U3310
##**Проверила:** Желтова Кристина Анатольевна

## **Загрузка данных**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

%matplotlib inline
sns.set_style("whitegrid")

# export KAGGLE_API_TOKEN=KGAT_e32f1e27c3076e0083f2ef1a7904bc58
%pip install opendatasets -q
import opendatasets as od

od.download("https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset/data")

csv_files = glob.glob("ibm-hr-analytics-attrition-dataset/*.csv")

df = pd.read_csv(csv_files[0])

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: bulk2403
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset


100%|██████████| 50.1k/50.1k [00:00<00:00, 1.33MB/s]

### **Зафиксируем `random_state`**

In [12]:
import random
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Для воспроизводимости решения был зафиксирован параметр `random_state`, а также установлены одинаковые значения генератора случайных чисел в Python и NumPy.

## **1. Разбиение на обучающую и тестовую выборки**

In [13]:
X = df.drop(columns=["Attrition"])
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

print("\nРаспределение классов в train:")
print(y_train.value_counts(normalize=True).round(3))

print("\nРаспределение классов в test:")
print(y_test.value_counts(normalize=True).round(3))

Размер обучающей выборки: (1176, 34)
Размер тестовой выборки: (294, 34)

Распределение классов в train:
Attrition
No     0.838
Yes    0.162
Name: proportion, dtype: float64

Распределение классов в test:
Attrition
No     0.84
Yes    0.16
Name: proportion, dtype: float64


Для дальнейшего обучения модели данные были разделены на обучающую и тестовую выборки в соотношении 80/20.  
Так как целевая переменная является бинарной и классы в датасете несбалансированы, при разбиении была использована стратификация по целевой переменной. Это позволяет сохранить исходное соотношение классов в обеих выборках и получить более корректную оценку качества модели.

## **2. Константное предсказание**

### *2.1 Модель*

In [14]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

dummy_model = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy_model.fit(X_train, y_train)

DummyClassifier(random_state=42, strategy='most_frequent')

Был использован `DummyClassifier` со стратегией `most_frequent`, который всегда предсказывает наиболее частотный класс. В данном наборе данных таким классом является `No`, то есть сотрудник не уволится. Использование такой модели позволяет получить минимальный ориентир качества, с которым затем можно сравнивать более сложные алгоритмы.

### *2.2 Оценка качества*

In [15]:
y_pred_dummy = dummy_model.predict(X_test)

dummy_accuracy = accuracy_score(y_test, y_pred_dummy)
dummy_balanced_accuracy = balanced_accuracy_score(y_test, y_pred_dummy)
dummy_precision = precision_score(y_test, y_pred_dummy, pos_label="Yes", zero_division=0)
dummy_recall = recall_score(y_test, y_pred_dummy, pos_label="Yes", zero_division=0)
dummy_f1 = f1_score(y_test, y_pred_dummy, pos_label="Yes", zero_division=0)

print("Качество константного бейзлайна:")
print(f"Accuracy: {dummy_accuracy:.3f}")
print(f"Balanced Accuracy: {dummy_balanced_accuracy:.3f}")
print(f"Precision: {dummy_precision:.3f}")
print(f"Recall: {dummy_recall:.3f}")
print(f"F1-score: {dummy_f1:.3f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred_dummy, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_dummy))

Качество константного бейзлайна:
Accuracy: 0.840
Balanced Accuracy: 0.500
Precision: 0.000
Recall: 0.000
F1-score: 0.000

Classification report:
              precision    recall  f1-score   support

          No       0.84      1.00      0.91       247
         Yes       0.00      0.00      0.00        47

    accuracy                           0.84       294
   macro avg       0.42      0.50      0.46       294
weighted avg       0.71      0.84      0.77       294


Confusion matrix:
[[247   0]
 [ 47   0]]


Полученные результаты демонстрируют типичную проблему константного бейзлайна при несбалансированных классах.

Несмотря на достаточно высокое значение `accuracy` (0.840), данная метрика вводит в заблуждение. Модель всегда предсказывает наиболее частотный класс (в данном случае — "No"), игнорируя класс "Yes". В результате большая часть объектов классифицируется "правильно" только за счёт дисбаланса данных.

Это подтверждается значением `balanced accuracy = 0.5`, что соответствует случайному угадыванию и указывает на отсутствие реальной способности модели различать классы.

Метрики `precision`, `recall` и `F1-score` для класса "Yes" равны нулю, так как модель ни разу не предсказала данный класс. Это означает, что модель полностью бесполезна для решения исходной задачи, если целью является выявление объектов класса "Yes".

Таким образом, константный бейзлайн может использоваться только как нижняя граница качества. Любая осмысленная модель должна превосходить его, в первую очередь по метрике `balanced accuracy` и способности корректно предсказывать редкий класс.

## **3. Логистическая регрессия**

### *3.1 Модель*

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# Простая бейзлайновая модель
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        solver="liblinear"
    ))
])

baseline_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'DailyRate',
                                                   'DistanceFromHome',
                                                   'Education', 'EmployeeCount',
                                                   'EmployeeNumber',
                                                   'EnvironmentSatisfaction',
                                                   'HourlyRate',
                                                   'JobInvolvement', 'JobLevel',
                                                   'JobSatisfaction',
                                                   'MonthlyIncome',
                                                   'Mon...
                                                   'YearsWithCurrManager']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['BusinessTravel',
                                                   'Department',
                                                   'EducationField', 'Gender',
                                                   'JobRole', 'MaritalStatus',
                                                   'Over18', 'OverTime'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42, solver='liblinear'))])

В качестве бейзлайновой модели была выбрана логистическая регрессия — модель из семейства линейных методов. Она подходит для получения первого содержательного ориентира качества в задаче бинарной классификации.

Перед обучением была выполнена предобработка данных. Категориальные признаки были закодированы с помощью `OneHotEncoder`, поскольку они не имеют естественного порядка, а числовые признаки были стандартизованы с помощью `StandardScaler`. Пропуски в числовых и категориальных признаках были обработаны с помощью `SimpleImputer`.

Так как классы в целевой переменной несбалансированы, при обучении был использован параметр `class_weight='balanced'`. Это позволяет уменьшить смещение модели в сторону наиболее частотного класса и лучше учитывать редкий целевой класс `Yes`.

### *3.2 Оценка качества*

In [11]:
y_pred_baseline = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_balanced_accuracy = balanced_accuracy_score(y_test, y_pred_baseline)
baseline_precision = precision_score(y_test, y_pred_baseline, pos_label="Yes", zero_division=0)
baseline_recall = recall_score(y_test, y_pred_baseline, pos_label="Yes", zero_division=0)
baseline_f1 = f1_score(y_test, y_pred_baseline, pos_label="Yes", zero_division=0)

print("Качество бейзлайновой модели:")
print(f"Accuracy: {baseline_accuracy:.3f}")
print(f"Balanced Accuracy: {baseline_balanced_accuracy:.3f}")
print(f"Precision: {baseline_precision:.3f}")
print(f"Recall: {baseline_recall:.3f}")
print(f"F1-score: {baseline_f1:.3f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred_baseline, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_baseline))

Качество бейзлайновой модели:
Accuracy: 0.745
Balanced Accuracy: 0.702
Precision: 0.341
Recall: 0.638
F1-score: 0.444

Classification report:
              precision    recall  f1-score   support

          No       0.92      0.77      0.83       247
         Yes       0.34      0.64      0.44        47

    accuracy                           0.74       294
   macro avg       0.63      0.70      0.64       294
weighted avg       0.83      0.74      0.77       294


Confusion matrix:
[[189  58]
 [ 17  30]]


После обучения модель была проверена на отложенной тестовой выборке. Качество оценивалось по ранее выбранным метрикам: `accuracy`, `balanced accuracy`, `precision`, `recall` и `F1-score`.

Основной метрикой для сравнения моделей в данной задаче является `balanced accuracy`, так как она учитывает качество предсказаний по каждому классу и более корректна при дисбалансе классов.

На отложенной выборке бейзлайновая модель показала заметно более высокое качество по сравнению с константным предсказанием.

Значение `accuracy` равно 0.745, однако в условиях дисбаланса классов более показательна метрика `balanced accuracy`, которая составила 0.702. Это означает, что модель в целом умеет различать классы лучше, чем случайное угадывание и значительно лучше, чем константный бейзлайн, для которого `balanced accuracy` была равна 0.500.

Для целевого класса `Yes` модель достигла `precision = 0.341`, `recall = 0.638` и `F1-score = 0.444`. Это означает, что модель находит большую часть объектов класса `Yes`, но при этом допускает заметное число ложноположительных предсказаний. Иными словами, она скорее склонна чаще предсказывать класс `Yes`, чем пропускать его, что повышает полноту, но снижает точность.

Матрица ошибок показывает, что из 47 объектов класса `Yes` модель правильно распознала 30, а 17 отнесла к классу `No`. При этом из 247 объектов класса `No` 189 были классифицированы верно, а 58 ошибочно отнесены к классу `Yes`. Таким образом, модель уже извлекает полезный сигнал из признаков.

## **4. Общий вывод**

В рамках данного задания были построены и проанализированы два типа бейзлайновых решений: константное предсказание и простая модель из семейства линейных методов (логистическая регрессия).

Сначала был построен константный бейзлайн, который всегда предсказывает наиболее частотный класс `No`. Несмотря на высокое значение `accuracy` (0.840), такая модель не решает бизнес-задачу: она полностью игнорирует сотрудников, относящихся к классу `Yes`. Это подтверждается тем, что `precision`, `recall` и `F1-score` для класса `Yes` равны нулю, а `balanced accuracy` составляет 0.500. Следовательно, константный бейзлайн можно рассматривать только как нижнюю границу качества, но не как практическое решение задачи.

Далее была обучена бейзлайновая модель из простого семейства — логистическая регрессия — с учётом предобработки признаков и дисбаланса классов. На отложенной выборке эта модель показала заметно более содержательные результаты: `balanced accuracy` выросла до 0.702, а для класса `Yes` были получены `precision = 0.341`, `recall = 0.638` и `F1-score = 0.444`. Это означает, что модель уже способна находить значительную часть сотрудников, находящихся в зоне риска увольнения, хотя и допускает заметное число ложноположительных срабатываний.

Таким образом, логистическая регрессия оказалась корректным бейзлайном для данной задачи: она действительно извлекает полезный сигнал из данных о сотрудниках и заметно превосходит константное предсказание. Полученные результаты подтверждают, что в данных присутствуют закономерности, позволяющие прогнозировать attrition.